# Kaggle: predict + submit (0.942-pipeline port)

Attach as Datasets: the competition data, `cell-tracking-src` (the zip from `scripts/package_for_kaggle.py`: code + `models/primary.pth`, `models/secondary.pth`, `models/deepcenter.pt` + `ARTIFACT_MANIFEST.json`), and `biohub-tracking-wheels` (= `context/pack_primary/wheels/`, the offline wheels for tracksdata / ilpy / pyscipopt / polars / ...). This notebook runs with internet OFF. Needs a GPU accelerator.

In [ ]:
import subprocess, sys
from pathlib import Path

WHEELS_DATASET = Path('/kaggle/input/biohub-tracking-wheels')  # adjust to the attached Dataset's mount name
SRC_DATASET = Path('/kaggle/input/cell-tracking-src')          # adjust to the attached Dataset's mount name
assert SRC_DATASET.exists(), f'code Dataset not attached at {SRC_DATASET}'
assert WHEELS_DATASET.exists(), f'wheels Dataset not attached at {WHEELS_DATASET}'

wheel_dirs = sorted({p.parent for p in WHEELS_DATASET.rglob('*.whl')})
assert wheel_dirs, f'no .whl files under {WHEELS_DATASET}'
find_links = []
for d in wheel_dirs:
    find_links += ['--find-links', str(d)]
# Offline install of the ILP stack (+ blosc2/zarr if the image lacks them). Do not
# install the bundled geff wheel if a newer geff is preinstalled.
pkgs = ['tracksdata', 'ilpy', 'pyscipopt', 'polars', 'polars_runtime_32', 'rustworkx',
        'sqlalchemy', 'dask', 'imagecodecs', 'pyarrow', 'blosc2', 'zarr']
subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-index', *find_links, *pkgs], check=True)
import os
os.environ['POLARS_PREFER_PKG'] = '32'
import tracksdata, ilpy, pyscipopt, blosc2, zarr
print('ILP stack ok:', tracksdata.__version__)


In [ ]:
import json, os
sys.path.insert(0, str(SRC_DATASET / 'src'))
SRC_ENV = {**os.environ, 'PYTHONPATH': str(SRC_DATASET / 'src'), 'POLARS_PREFER_PKG': '32'}

manifest = json.loads((SRC_DATASET / 'ARTIFACT_MANIFEST.json').read_text())
print('artifact manifest:', json.dumps(manifest, indent=1)[:3000])
models = manifest['models']
assert 'primary' in models, 'no primary model in this artifact'
PRIMARY = SRC_DATASET / models['primary']['path']
SECONDARY = SRC_DATASET / models['secondary']['path'] if 'secondary' in models else None
DEEPCENTER = SRC_DATASET / models['deepcenter']['path'] if 'deepcenter' in models else None
for p in (PRIMARY, SECONDARY, DEEPCENTER):
    assert p is None or p.exists(), p
import hashlib
for key, entry in models.items():
    digest = hashlib.sha256((SRC_DATASET / entry['path']).read_bytes()).hexdigest()
    assert digest == entry['sha256'], f'{key}: sha256 mismatch -- wrong weights attached'
print('weights verified:', {k: v['sha256'][:12] for k, v in models.items()})


In [ ]:
from cell_tracking import config
print('on_kaggle:', config.on_kaggle())
print('test_dir:', config.get_test_dir())


In [ ]:
cmd = [sys.executable, str(SRC_DATASET / 'scripts' / 'predict.py'),
       '--checkpoint', str(PRIMARY), '--out-dir', '/kaggle/working/preds',
       '--dump-stats', '/kaggle/working/predict_stats.json']
if SECONDARY is not None:
    cmd += ['--secondary-checkpoint', str(SECONDARY)]
if DEEPCENTER is not None:
    cmd += ['--deepcenter', str(DEEPCENTER)]
print(' '.join(cmd))
subprocess.run(cmd, check=True, env=SRC_ENV)


In [ ]:
subprocess.run([sys.executable, str(SRC_DATASET / 'scripts' / 'make_submission.py'),
                '--geff-dir', '/kaggle/working/preds', '--out', 'submission.csv'], check=True, env=SRC_ENV)


In [ ]:
# Schema / topology audit (the 0.942 notebook's rules): contiguous ids, dt=1, in-degree<=1, out-degree<=2, no negatives.
import csv
from collections import defaultdict
rows = list(csv.DictReader(open('submission.csv')))
assert [int(r['id']) for r in rows] == list(range(len(rows))), 'ids not contiguous'
by_ds = defaultdict(lambda: {'t': {}, 'in': defaultdict(int), 'out': defaultdict(int)})
for r in rows:
    d = by_ds[r['dataset']]
    if r['row_type'] == 'node':
        assert min(int(r['t']), int(r['z']), int(r['y']), int(r['x'])) >= 0, r
        d['t'][int(r['node_id'])] = int(r['t'])
for r in rows:
    if r['row_type'] == 'edge':
        d = by_ds[r['dataset']]
        s, t = int(r['source_id']), int(r['target_id'])
        assert s in d['t'] and t in d['t'], 'dangling edge'
        assert d['t'][t] == d['t'][s] + 1, 'edge not dt=1'
        d['in'][t] += 1; d['out'][s] += 1
for name, d in by_ds.items():
    assert max(d['in'].values(), default=0) <= 1, f'{name}: in-degree > 1'
    assert max(d['out'].values(), default=0) <= 2, f'{name}: out-degree > 2'
print('audit ok:', len(rows), 'rows,', len(by_ds), 'datasets')
